# L30 Lab: CFAR Processing, notebook

This notebook runs CFAR on the course's recorded data, or on your own Save Capture files. You do not write the detector; you run it, change what each task names, and record what you see in Tables A to F of the L30 Lab packet. There are three tasks, (a) to (c), about 30 to 45 minutes in all, and seven changed lines between them.

**To run it:** unzip the L30 Lab bundle (`L30_lab.zip`, linked from the page's Notebook Setup frame) into one folder: this notebook, `radar_lab.py`, and every `L30_*.npz`. Run the cells in order from the top.

The data comes from two of the Phaser app's presets:

- **Range (CFAR)**, the preset the app's CFAR overlay runs on: 0.5 ms chirps of 500 MHz at 10 GHz, 2 MSPS, one chirp a display update, 0.30 m range cells. `L30_range_person.npz`, `L30_walker.npz`, and `L30_range_empty.npz` are what Save Capture writes on it, and the No Kit cells below use them for Procedures A to C.
- **Range + Velocity**, the L29 burst: 256 chirps of 300 µs, 0.5 ms apart, 500 MHz at 9.9 GHz, 2 MSPS, so the map's cells are 0.30 m by 0.118 m/s. `L30_empty.npz`, `L30_wall_hand.npz`, and `L30_weak_strong.npz` are bursts of it, and tasks (a) to (c) run 2-D CFAR on their maps.

To run a cell on your own capture instead, change the file name in the next cell.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage, stats
import radar_lab as rl

# Where the captures are: this folder (the unzipped bundle), or the course's
# radar-data/ if this notebook sits beside it.
DATA = Path(".") if Path("L30_empty.npz").exists() else Path("../radar-data")

# Range (CFAR) preset: one range profile a display update (Procedures A to C)
PERSON_FILE = "L30_range_person.npz"     # Procedure A: you standing still at 3 m
WALKER_FILE = "L30_walker.npz"           # Procedure B: the walk
EMPTY_FILE = "L30_range_empty.npz"       # Procedure C: the empty scene
# Range + Velocity preset: one 256-chirp burst each (tasks (a) to (c))
empty = rl.load_capture(DATA / "L30_empty.npz")              # task (a): no target
wall_hand = rl.load_capture(DATA / "L30_wall_hand.npz")      # tasks (b) and (c)
weak_strong = rl.load_capture(DATA / "L30_weak_strong.npz")  # task (b)

for name in (PERSON_FILE, WALKER_FILE, EMPTY_FILE, "L30_empty.npz", "L30_wall_hand.npz", "L30_weak_strong.npz"):
    cap = rl.load_capture(DATA / name)
    print(f"{name:22s} {cap['iq'].shape[0]:3d} chirps of {cap['iq'].shape[1]:4d} samples, "
          f"{cap['fc']/1e9:.1f} GHz, T = {cap['T_ramp']*1e6:.0f} us, PRI = {cap['PRI']*1e3:.1f} ms, "
          f"f0 = {cap['f0']/1e3:.1f} kHz")

In [ ]:
# Helpers used by the tasks. Run this cell; you do not need to change it.
def n_ref(guard, ref):
    """Reference cells in cfar_2d's ring for (range, Doppler) half-widths."""
    (gr, gv), (rr, rv) = guard, ref
    return (2 * (gr + rr) + 1) * (2 * (gv + rv) + 1) - (2 * gr + 1) * (2 * gv + 1)

def quiet_cells(rng, vel, guard, ref):
    """Cells that can only hold noise in an empty scene: beyond 1 m, and far
    enough from 0 m/s that no reference cell reaches the kit's own leakage."""
    j0 = np.argmin(np.abs(vel))
    far_from_zero = np.abs(np.arange(vel.size) - j0) > guard[1] + ref[1] + 1
    return (rng > 1.0)[:, None] & far_from_zero[None, :]

def targets(det, hits):
    """One entry per cluster of touching detected cells: its strongest cell and
    the cluster's size, as (range, velocity, snr_db, n_cells), strongest first."""
    lab, _ = ndimage.label(det)
    best = {}
    for (i, j), h in zip(zip(*np.nonzero(det)), hits):     # hits run in this order
        k = lab[i, j]
        if k not in best or h[2] > best[k][2]:
            best[k] = (*h, int((lab == k).sum()))
    return sorted(best.values(), key=lambda t: -t[2])

def show(ax, rd, rng, vel, det, title, r_max=8.0):
    """The map in dBFS, with each detected cell marked."""
    keep = rng <= r_max
    ax.pcolormesh(vel, rng[keep], rd[keep], vmin=-120, vmax=-30, cmap="Blues", shading="auto")
    i, j = np.nonzero(det & keep[:, None])
    ax.plot(vel[j], rng[i], "s", mfc="none", mec="red", ms=4)
    ax.set_xlim(-4, 4); ax.set_xlabel("velocity (m/s), + closing"); ax.set_ylabel("range (m)")
    ax.set_title(title, fontsize=10)

## No kit? Procedures A to C on recorded data

With a kit, you fill Tables A and B from the app, and Table C from the Procedure C cell run on your own empty-scene capture: set `EMPTY_FILE` in the first cell to its name, and `G1, R1, BIAS1` in the Procedure A cell to your app setting, and run both cells. The cells take any Save Capture file from the Range (CFAR) preset. Without a kit, use the app's **Load Recording** on the same three files to watch each procedure, and read the numbers for Tables A to C from these cells.

The app's CFAR works on one range profile at a time: one chirp, one display update. `app_profile` and `threshold_1d` compute exactly what the app draws: a Blackman window, the ranges from 0 to 20 m (the app's Range Max), and cell averaging in power with Guard Cells and Ref Cells counted on **each** side, the reference cells that exist used at the ends. That is the lesson's estimator, so the lesson's bias formula applies, with $N$ = 2 × Ref Cells.

In [ ]:
C = 3.00e8

def app_profile(cap, k, range_max=20.0):
    """Update k's range profile as the app draws it: (range_m, power_db)."""
    iq = cap["iq"][k]
    n = iq.size
    w = np.blackman(n)                                   # the app's FFT Window, Blackman
    X = np.fft.fft(iq * w)[: n // 2] / w.sum()
    r = (np.arange(n // 2) * cap["fs"] / n - cap["f0"]) * C * cap["T_ramp"] / (2 * cap["B"])
    keep = (r >= 0) & (r <= range_max)
    return r[keep], 20 * np.log10(np.abs(X[keep]) + 1e-15)

def threshold_1d(p_db, guard, ref, bias_db):
    """The app's CFAR threshold (dB) and detections: guard and ref cells on each side."""
    n, P = p_db.size, 10 ** (p_db / 10)
    cs, i = np.r_[0, np.cumsum(P)], np.arange(p_db.size)
    lo, hi = np.maximum(i - guard - ref, 0), np.minimum(i + guard + ref + 1, n)
    glo, ghi = np.maximum(i - guard, 0), np.minimum(i + guard + 1, n)
    noise = ((cs[hi] - cs[lo]) - (cs[ghi] - cs[glo])) / ((hi - lo) - (ghi - glo))
    thr = 10 * np.log10(noise) + bias_db
    return thr, p_db > thr

def crossings(det, r):
    """The center range of each run of touching detected cells."""
    edges = np.flatnonzero(np.diff(np.r_[0, det.astype(int), 0]))
    return [(r[a] + r[b - 1]) / 2 for a, b in zip(edges[::2], edges[1::2])]

### Procedure A: place the threshold

The cell draws update 0 of `L30_range_person.npz`, a person standing still at 3.0 m in front of a wall at 6.0 m, with the threshold over it. The leakage near 0 m and the wall are real echoes that cross any CFAR threshold; the clutter you are placing the threshold over is the noise and the skirts between them.

1. Run it as it stands, at the app's opening setting: 4 guard and 16 reference cells and a 6 dB bias. Then set the lesson's 1D window, `G1, R1, BIAS1 = 2, 16, 12.4`, and run it again. Find in the plot what raised the threshold under you.
2. Change `G1`, `R1`, and `BIAS1` until your peak is at least 3 dB over the threshold and the highest noise peak at least 3 dB under it. Record your setting and both margins in Table A.
3. Set `G1 = 0` with your other two settings, record the threshold's rise at your peak, and set it back.

**Check:** the app's opening setting misses you by 5.1 dB and puts a noise crossing in 15 of the 32 updates; the lesson's window misses you by 11.2 dB with none. Both windows reach the leakage, 20 dB stronger than you and 3 m away, inside 16 cells of 0.30 m. 3 guard and 4 reference cells at 13 dB meets both margins: you stand 39.3 dB over the threshold, and the highest noise peak sits 6.3 dB under it. Three guard cells cover the Blackman window's main lobe, which spreads a peak three cells each way. With no guard cells, your own main lobe enters the window, the threshold at your peak rises by 42.7 dB, and you are missed.

In [ ]:
G1, R1, BIAS1 = 4, 16, 6.0      # PROCEDURE A: Guard Cells, Ref Cells (each side), CFAR Bias (dB)

person = rl.load_capture(DATA / PERSON_FILE)
r, p = app_profile(person, 0)
near = np.abs(r - 3.0) < 1.0
ip = np.flatnonzero(near)[np.argmax(p[near])]              # your peak, near 3 m
objects = (r < 1.0) | (np.abs(r - r[ip]) < 1.0) | (np.abs(r - 6.0) < 1.0)   # leakage, you, wall
thr, det = threshold_1d(p, G1, R1, BIAS1)
fa = sum(bool((threshold_1d(app_profile(person, k)[1], G1, R1, BIAS1)[1] & ~objects).any())
         for k in range(person["iq"].shape[0]))
print(f"guard {G1}, reference {R1}, bias {BIAS1} dB:")
print(f"  your peak at {r[ip]:.2f} m: {p[ip] - thr[ip]:+.1f} dB over the threshold "
      f"({'found' if det[ip] else 'missed'})")
print(f"  highest noise or skirt peak: {np.max((p - thr)[~objects]):+.1f} dB over the threshold")
print(f"  crossings at {np.round(crossings(det, r), 2)} m")
print(f"  updates with a noise crossing: {fa} of {person['iq'].shape[0]}")
plt.figure(figsize=(8, 3))
plt.plot(r, p, lw=0.8, label="range profile"); plt.plot(r, thr, "g--", label="threshold")
plt.ylim(-120, -10); plt.legend(loc="upper right")
plt.xlabel("range (m)"); plt.ylabel("power (dBFS)"); plt.title(f"{PERSON_FILE}, update 0")
plt.grid(alpha=0.3); plt.show()

### Procedure B: apply it and walk

`L30_walker.npz` is the app's range profile while a person walks from 1.0 m out to 4.5 m and back at 1 m/s, toward a wall at 6.0 m: one update every 0.1 s, 70 updates in 7 s. The cell applies your Procedure A setting to every update, and prints the nearest and farthest range at which the threshold finds the walker on the way out, and the updates with any other crossing besides the leakage and the wall.

1. Run it with `EXTRA_DB = 0` and record the row in Table B.
2. Set `EXTRA_DB = 3`, run it again, and record the second row.

**Check:** at 3 guard, 4 reference cells, and 13 dB, the walker is found in 29 of the 70 updates, from 2.29 m to 3.39 m on the way out, with no other crossing. With 3 dB more, it is found in 27, over the same stretch. The walker is 30 to 60 dB over the noise the whole way, so the bias barely matters: the leakage enters the window inside about 2.3 m, and the wall at 6.0 m enters it beyond about 3.6 m. Part 3 asks what that means for Question 4.

In [ ]:
EXTRA_DB = 0                     # PROCEDURE B: change to 3 for the second row

walker = rl.load_capture(DATA / WALKER_FILE)
FIXED = [(0.0, 1.0), (5.0, 7.0)]    # the leakage and the wall: echoes that never move
n_upd = walker["iq"].shape[0]
found, others, rows = 0, 0, []
for k in range(n_upd):
    r, p = app_profile(walker, k)
    _, det = threshold_1d(p, G1, R1, BIAS1 + EXTRA_DB)
    xs = [x for x in crossings(det, r) if not any(a <= x <= b for a, b in FIXED)]
    near = [x for x in xs if x < 5.0]           # the walker's stretch of floor
    found += bool(near); others += bool(set(xs) - set(near))
    rows.append((k * walker["PRI"], near))
seen = [x for t, near in rows for x in near]
span = f"from {min(seen):.2f} m to {max(seen):.2f} m" if seen else "never found"
print(f"bias {BIAS1 + EXTRA_DB:.1f} dB: walker found in {found} of {n_upd} updates, {span}; "
      f"{others} updates with another crossing")
plt.figure(figsize=(8, 3.5))
w_db = np.array([app_profile(walker, k)[1] for k in range(n_upd)])
r = app_profile(walker, 0)[0]
t = np.arange(n_upd) * walker["PRI"]
keep = r <= 8
plt.pcolormesh(r[keep], t, w_db[:, keep], vmin=-100, vmax=-20, cmap="Blues", shading="auto")
for tk, near in rows:
    plt.plot(near, [tk] * len(near), "s", mfc="none", mec="red", ms=4)
plt.xlabel("range (m)"); plt.ylabel("time (s)")
plt.title(f"{WALKER_FILE}, detections at {BIAS1 + EXTRA_DB:.1f} dB"); plt.show()

### Procedure C: false alarms on an empty scene

The first 30 updates of `L30_range_empty.npz`, or of your own empty-scene capture, are 30 updates of the app with the beam clear. The cell uses the setting in the Procedure A cell, so set that to yours first. The cell counts the updates in which anything beyond 1 m crosses the threshold, at your setting and 3 and 6 dB below it. Record the three rows in Table C.

**Check:** at 3 guard, 4 reference cells, and 13 dB, 1 of the 30 updates holds a crossing; at 10 dB, 3; at 7 dB, 20, with 31 crossings in all. The crossings are single cells, spread from 5 m to 20 m at ranges that change from update to update, which is what noise looks like. Pre-Lab Question 1 predicts about 0.1, 3, and 22 of the 30 updates for 4 reference cells a side.

In [ ]:
empty_r = rl.load_capture(DATA / EMPTY_FILE)
for drop in (0, 3, 6):
    hit, total, where = 0, 0, []
    for k in range(30):
        r, p = app_profile(empty_r, k)
        _, det = threshold_1d(p, G1, R1, BIAS1 - drop)
        xs = [x for x in crossings(det, r) if x > 1.0]
        hit += bool(xs); total += len(xs); where += xs
    print(f"bias {BIAS1 - drop:5.1f} dB: a crossing in {hit:2d} of 30 updates, {total} crossings, "
          f"e.g. at {np.round(where[:6], 1)} m")

## Task (a): set the bias for a false-alarm rate, then count

`L30_empty.npz` holds no target, only noise and the kit's own leakage at 0 m. Every detection in it is a false alarm, apart from the leakage itself, so the cell counts detections only where no reference cell can reach the leakage: beyond 1 m and away from 0 m/s. Those are 64 106 cells.

1. **Change one line:** replace `None` in `BIAS_DB = None` with the bias, in dB, that gives `PFA` with `N` reference cells (the bias equation of the lesson's "Bias From the False-Alarm Probability" frame; your Pre-Lab Question 1). Run the cell with `PFA = 1e-3` and record the design count, the measured count, and the measured rate in Table D.
2. **Change one line:** set `PFA = 1e-4`, run it, and record the second row.
3. **Change one line:** set `WINDOW = None`, the rectangular window, with `PFA` back at `1e-3`, and record the third row.

**Check:** `BIAS_DB` prints as 8.50 dB at $10^{-3}$ and 9.78 dB at $10^{-4}$. The design counts are 64 and 6. With the Hann window the measured rate comes out above the design: 91 false alarms, 1.42 times it, at $10^{-3}$, and 9, 1.40 times it, at $10^{-4}$. With the rectangular window at $10^{-3}$ it lands on the design: 63, 0.98 times it. Part 3 of the packet asks you why the window changes the count when nothing else did. A count of a few has a scatter of about its own square root, so read the $10^{-4}$ row with that in mind.

In [ ]:
GUARD = (2, 2)           # guard cells each side, (range, Doppler)
REF = (4, 4)             # reference cells each side beyond the guard cells, (range, Doppler)
PFA = 1e-3               # TASK (a), step 2: change to 1e-4
WINDOW = "hann"          # TASK (a), step 3: change to None, with PFA back at 1e-3
N = n_ref(GUARD, REF)
BIAS_DB = None           # TASK (a), step 1: the bias for PFA with N reference cells, in dB

rd, rng, vel = rl.range_doppler(empty, window=WINDOW)
test = quiet_cells(rng, vel, GUARD, REF)
if BIAS_DB is None:
    print(f"N = {N} reference cells. Set BIAS_DB (step 1) and run this cell again.")
else:
    det, _ = rl.cfar_2d(rd, GUARD, REF, BIAS_DB)
    n_fa, n_test = int(det[test].sum()), int(test.sum())
    print(f"window {WINDOW}, N = {N}, BIAS_DB = {BIAS_DB:.2f} dB, design PFA {PFA:.0e}")
    print(f"  {n_test} cells tested: design {PFA * n_test:.0f} false alarms, measured {n_fa}, "
          f"rate {n_fa / n_test:.2e} ({n_fa / n_test / PFA:.2f} x design)")
    fig, ax = plt.subplots(figsize=(8, 3.5))
    show(ax, rd, rng, vel, det & test, f"L30_empty: false alarms at design PFA {PFA:.0e}", r_max=40)
    ax.set_xlim(vel[0], vel[-1]); plt.show()

## Task (b): cell averaging, greatest-of, smallest-of

Two scenes, the same detector settings: 2 guard and 4 reference cells each way and an 11.6 dB bias, the lesson's 2D example. The bias is held fixed so that only the method changes; strictly, each method needs its own bias for the same $P_{fa}$ (the lesson's "Greatest-Of and Smallest-Of" frame).

- `L30_wall_hand.npz`: a wall at 5.0 m, square to the beam, and a hand waving about 2.4 m; it spans about 2.3 to 2.6 m during the burst. The wall stands still, so it sits at 0 m/s with the leakage.
- `L30_weak_strong.npz`: a strong mover at 4.2 m and a weak one at 3.0 m, four range cells nearer, at almost the same velocity: the lesson's masking case.

The cell prints one line per cluster of detected cells: its strongest cell's range, velocity, and SNR over its own noise estimate, and how many cells the cluster holds.

1. Run it as it stands (`METHOD = "ca"`). For each scene, record in Table E the clusters at the wall (or the strong target), at the hand (or the weak target), and anywhere else.
2. **Change one line:** set `METHOD = "go"` and run it again. Then set `METHOD = "so"` and run it a third time.

**Check:** with `"ca"` the leakage is a cluster of 8 cells, the hand 6, and the wall 9, 3 range by 3 velocity cells; in the second scene the strong target is 7 cells, the leakage 9, and the weak target is missing. Neither scene has a false alarm. `"go"` gives the same clusters, and still misses the weak target. `"so"` finds the weak target, 7 cells near 3.2 m, but the wall's cluster grows from 9 cells to 20, the hand's from 6 to 21, and the strong target's from 7 to 22, and three clusters of false alarms appear in the second scene: 3 cells beside the leakage at 0.47 m and $+4.62$ m/s, and single cells near 17.3 m and 80.3 m. Each line reports the cluster's cell with the highest SNR, which is not always its strongest cell, so the ranges move between methods: the hand reads 2.57 m under `"ca"` and 2.27 m under `"so"`. Look at where the extra cells sit in the plot; Part 3 asks you to explain each change.

In [ ]:
METHOD = "ca"            # TASK (b): change to "go", then to "so"

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, cap) in zip(axs, (("L30_wall_hand", wall_hand), ("L30_weak_strong", weak_strong))):
    rd, rng, vel = rl.range_doppler(cap)
    det, hits = rl.cfar_2d(rd, (2, 2), (4, 4), 11.6, axes=(rng, vel), method=METHOD)
    print(f"{name}, method {METHOD}: {int(det.sum())} cells detected, in these clusters:")
    for r, v, snr, n in sorted(targets(det, hits)):
        print(f"   {r:6.2f} m  {v:+6.2f} m/s  {snr:5.1f} dB  {n:3d} cells")
    show(ax, rd, rng, vel, det, f"{name}, method {METHOD}")
plt.tight_layout(); plt.show()

## Task (c): the probability of detection, and the list for L31

The hand in `L30_wall_hand.npz` stands far above the noise, so CFAR finds it at any bias you would use: one map says $P_d = 1$ and nothing else. $P_d$ only moves at an SNR near the 13 dB of Lesson 27, and measuring it takes many maps of the same target. The cell below makes them: it adds fresh noise to the capture `TRIALS` times, enough to bring the hand's strongest cell down to `SNR_DB` above the noise, as if the hand stood much farther away, and counts the maps in which `cfar_2d` still declares that cell.

Only the noise changes from map to map; the hand's echo is the same recorded one every time. That is a **steady** target, so the prediction to compare with is Lesson 27's steady-target (Marcum) $P_d$. The cell prints it twice, with the noise power known and with it estimated from $N$ independent reference cells, which includes the CFAR loss. It also prints the Swerling 1 value, Pre-Lab Question 2's fluctuating target under the same CFAR, for contrast: a real hand's echo changes from map to map, and then that is the number to expect.

1. Run it as it stands, at `BIAS_DB_C = 11.6`, the bias for $10^{-6}$. Record the measured $P_d$ and the predictions in Table F.
2. **Change one line:** set `BIAS_DB_C` to your task (a) bias for $10^{-3}$ and run it again. Record the second row.

**Check:** at 12 dB the measured $P_d$ is 0.60 at the $10^{-6}$ bias, against 0.63 steady (both columns) and 0.43 Swerling 1; at the $10^{-3}$ bias, 8.50 dB, it is 0.98, against 0.98 and 0.97 steady and 0.66 Swerling 1. Each measurement is within the scatter of the steady prediction: with 50 maps, a measured $P_d$ has a scatter of about 0.07. The cell takes about half a minute.

In [ ]:
BIAS_DB_C = 11.6         # TASK (c), step 2: change to your task (a) bias for 1e-3
SNR_DB = 12.0            # the hand's strongest cell over the noise, after the added noise
TRIALS = 50

cap = wall_hand
rd0, rng, vel = rl.range_doppler(cap)
hand = ((rng > 1.5) & (rng < 3.5))[:, None] & (np.abs(vel) > 0.3)[None, :]
ih, jh = np.unravel_index(np.argmax(np.where(hand, rd0, -300)), rd0.shape)

# Noise per sample that puts the hand's cell SNR_DB above the noise in its cell.
# The two Hann FFTs pass 1.5/n of the noise power each (L28, the window's noise bandwidth).
m, n = cap["iq"].shape
p_cell = 10 ** ((rd0[ih, jh] - SNR_DB) / 10)
sigma = np.sqrt(p_cell / (1.5 / n * 1.5 / m) / 2)           # per real and imaginary part
rng_noise = np.random.default_rng(30)
found = 0
for _ in range(TRIALS):
    noisy = {**cap, "iq": cap["iq"] + sigma * (rng_noise.standard_normal((m, n))
                                               + 1j * rng_noise.standard_normal((m, n)))}
    rd, _, _ = rl.range_doppler(noisy)
    det, _ = rl.cfar_2d(rd, (2, 2), (4, 4), BIAS_DB_C)
    found += bool(det[ih, jh])

# Lesson 27's steady-target Pd (Marcum Q), with the noise known and with CA-CFAR's
# estimate from N independent cells: the threshold alpha * x, x the estimate over Pn.
alpha, snr, N = 10 ** (BIAS_DB_C / 10), 10 ** (SNR_DB / 10), n_ref((2, 2), (4, 4))
pd_known = stats.ncx2.sf(2 * alpha, 2, 2 * snr)
x = np.linspace(0.3, 2.5, 2001)
pd_cfar = np.sum(stats.ncx2.sf(2 * alpha * x, 2, 2 * snr) * stats.gamma.pdf(x, N, scale=1 / N)) * (x[1] - x[0])
# For contrast, Pre-Lab Question 2's fluctuating target (Swerling 1) under the same CA-CFAR.
pd_swerling = (1 + alpha / (N * (1 + snr))) ** -N
Pd_measured = found / TRIALS
print(f"hand cell {rng[ih]:.2f} m, {vel[jh]:+.2f} m/s, {rd0[ih, jh]:.1f} dBFS; set to {SNR_DB:.0f} dB SNR")
print(f"BIAS_DB_C = {BIAS_DB_C:.2f} dB: found in {found} of {TRIALS} maps, Pd = {Pd_measured:.2f}")
print(f"  steady target (the one to compare): noise known Pd = {pd_known:.2f};  CA-CFAR, N = {N}: Pd = {pd_cfar:.2f}")
print(f"  fluctuating target (Swerling 1), CA-CFAR, N = {N}: Pd = {pd_swerling:.2f}")

### The detection list you keep for L31

The last cell makes a detection list: one entry per object, from the wall-and-hand burst or your own, after the 2-pulse canceller of Lesson 29 has removed what stands still. As shipped it prints the list in cell indices.

3. **Task (c), step 3: change one line.** Add `axes=(rng, vel)` to the `cfar_2d` call, so the list comes back in meters and meters per second, and run it. It writes `L30_detections.csv` beside this notebook.

**Check:** one entry, the hand, at 2.57 m and $-0.83$ m/s, 45.9 dB over its noise estimate, 10 cells: opening, because the burst catches the hand moving away from the kit. The wall and the leakage are gone with the canceller. Record it in Table F and mark it as the target. On your own burst the list may hold false alarms too, usually single cells just over the bias; mark them in Table F, and leave them in the file. The CSV keeps every entry, false alarms included: it is your record of what this chain found. The L31 Lab runs the same chain again on its own records, so nothing there reads this file.

In [ ]:
LIST_CAPTURE = wall_hand        # or rl.load_capture("your_burst.npz")

rd, rng, vel = rl.range_doppler(rl.mti(LIST_CAPTURE, "2pulse"))
det, hits = rl.cfar_2d(rd, (2, 2), (4, 4), 11.6)   # TASK (c), step 3: add axes=(rng, vel)
found_list = targets(det, hits)
for r, v, snr, n in found_list:
    print(f"range {r:7.2f}   velocity {v:+7.2f}   SNR {snr:5.1f} dB   {n} cells")
np.savetxt("L30_detections.csv", [t[:3] for t in found_list], delimiter=",", fmt="%.3f",
           header="range_m,vel_mps,snr_db", comments="")

## Before you close

Save the notebook with your changes and your plots, and upload the `.ipynb` with the packet. Keep the task cells' names and variables as they are: a Gradescope autograder for the notebooks is planned, and it will read them. Keep the notebook, `L30_detections.csv`, `radar_lab.py`, and your captures together. Anything you do beyond these three tasks, such as another guard band, a scene of your own, or the bias each method needs for the same $P_{fa}$, belongs in your report's Extra Effort.